In [15]:
from onnx2pytorch import ConvertModel
from torch import nn
import torch.nn.functional as F
import onnx
import idx2numpy, numpy as np, torch
from torch.utils.data import DataLoader, TensorDataset

In [16]:
model_path = "D:\\automated-object-control\\lab-for-block2\\nn_model.onnx"
data_idx_path = "D:\\automated-object-control\\lab-for-block2\\trainingDatas.idx"
label_idx_path = "D:\\automated-object-control\\lab-for-block2\\trainingLabels.idx"
onnx_model = onnx.load(model_path)
model = ConvertModel(onnx_model)
model.train()

ConvertModel(
  (MatMul_sequential/dense_1/BiasAdd:0): Linear(in_features=4, out_features=64, bias=True)
  (Relu_sequential/dense_1/Relu:0): ReLU(inplace=True)
  (MatMul_output_layer): Linear(in_features=64, out_features=3, bias=True)
)

In [29]:
X_idx = idx2numpy.convert_from_file(data_idx_path) #expect shape (N,4) or (N,1,4)
Y_idx = idx2numpy.convert_from_file(label_idx_path) #expect shape (N,)

print("X shape:", X_idx.shape, "| dtype:", X_idx.dtype)
print("Y shape:", Y_idx.shape, "| dtype:", Y_idx.dtype)

# 2) make contiguous writable copies with desired dtypes
X = torch.tensor(np.array(X_idx, dtype=np.float32, copy=True))
Y = torch.tensor(np.array(Y_idx, dtype=np.int64,  copy=True))

print("X:", X.shape, X.dtype, " | Y:", Y.shape, Y.dtype)

train_dataset = TensorDataset(X,Y)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True)

X shape: (8006, 4) | dtype: >f4
Y shape: (8006,) | dtype: >i4
X: torch.Size([8006, 4]) torch.float32  | Y: torch.Size([8006]) torch.int64


In [34]:
criterion = nn.CrossEntropyLoss()
model.eval()
with torch.no_grad():
    logits = model(X[:256].float())
    ce = criterion(logits, Y[:256].long())
    pred = logits.argmax(1)
    acc = (pred == Y[:256]).float().mean().item()
print("CE(loss) first 256:", ce.item(), " | acc:", round(acc,4))

CE(loss) first 256: 0.2355390340089798  | acc: 0.9453


In [33]:
model.train()
criterion = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)

def robustness_property_loss(model, x, y, eps=0.01, steps=3, alpha=None):
    if alpha is None: alpha = eps/2
    x_adv = x.detach().clone().requires_grad_(True)
    for _ in range(steps):
        logits = model(x_adv)
        fy = logits.gather(1, y.view(-1,1))
        margins = logits - fy
        margins.scatter_(1, y.view(-1,1), float('-inf'))
        worst = margins.max(1).values.mean()
        grad, = torch.autograd.grad(worst, x_adv)
        with torch.no_grad():
            x_adv += alpha * torch.sign(grad)
            x_adv = torch.max(torch.min(x_adv, x + eps), x - eps)  # L∞ projection
            # clamp here to YOUR input range if needed (e.g., 0..1 or z-score bounds)
        x_adv.requires_grad_(True)
    logits = model(x_adv)
    fy = logits.gather(1, y.view(-1,1))
    margins = logits - fy
    margins.scatter_(1, y.view(-1,1), float('-inf'))
    return F.relu(margins.max(1).values).mean()

In [35]:
lambda_start, lambda_end = 0.1, 3.0
eps_start,    eps_end    = 0.003, 0.02
epochs = 10

for epoch in range(epochs):
    t = epoch / max(1, epochs-1)
    lam = lambda_start + (lambda_end - lambda_start)*t
    eps = eps_start + (eps_end - eps_start)*t
    for x, y in train_loader:        # ensure x has shape [B, 4] with correct scaling
        x, y = x.float(), y.long()
        logits = model(x)
        data_loss = criterion(logits, y)
        prop_loss = robustness_property_loss(model, x, y, eps=eps, steps=3)
        loss = data_loss + lam * prop_loss
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        print(f"epoch {epoch+1}/{epochs} loss={loss:.3f} λ={lam:.2f} ε={eps:.4f}")

epoch 1/10 loss=0.252 λ=0.10 ε=0.0030
epoch 1/10 loss=0.205 λ=0.10 ε=0.0030
epoch 1/10 loss=0.172 λ=0.10 ε=0.0030
epoch 1/10 loss=0.236 λ=0.10 ε=0.0030
epoch 1/10 loss=0.246 λ=0.10 ε=0.0030
epoch 1/10 loss=0.233 λ=0.10 ε=0.0030
epoch 1/10 loss=0.262 λ=0.10 ε=0.0030
epoch 1/10 loss=0.155 λ=0.10 ε=0.0030
epoch 1/10 loss=0.182 λ=0.10 ε=0.0030
epoch 1/10 loss=0.209 λ=0.10 ε=0.0030
epoch 1/10 loss=0.216 λ=0.10 ε=0.0030
epoch 1/10 loss=0.196 λ=0.10 ε=0.0030
epoch 1/10 loss=0.241 λ=0.10 ε=0.0030
epoch 1/10 loss=0.276 λ=0.10 ε=0.0030
epoch 1/10 loss=0.168 λ=0.10 ε=0.0030
epoch 1/10 loss=0.209 λ=0.10 ε=0.0030
epoch 1/10 loss=0.204 λ=0.10 ε=0.0030
epoch 1/10 loss=0.177 λ=0.10 ε=0.0030
epoch 1/10 loss=0.157 λ=0.10 ε=0.0030
epoch 1/10 loss=0.207 λ=0.10 ε=0.0030
epoch 1/10 loss=0.167 λ=0.10 ε=0.0030
epoch 1/10 loss=0.191 λ=0.10 ε=0.0030
epoch 1/10 loss=0.235 λ=0.10 ε=0.0030
epoch 1/10 loss=0.194 λ=0.10 ε=0.0030
epoch 1/10 loss=0.115 λ=0.10 ε=0.0030
epoch 1/10 loss=0.199 λ=0.10 ε=0.0030
epoch 1/10 l

In [32]:
model.eval()
with torch.no_grad():
    logits = model(X)
    pred = logits.argmax(1)
    acc = (pred == Y).float().mean().item()
print("train accuracy:", round(acc, 4))

train accuracy: 0.9635
